# CSIRO Biomass - Multitask NDVI/Height Training

GPU experiment: train one image model with auxiliary NDVI and height heads plus biomass heads. Saves checkpoints, OOF predictions, logs, metrics, and a submission.

In [ ]:
import os, sys, json, math, time, random, subprocess, pkgutil
from pathlib import Path
import numpy as np
import pandas as pd

if pkgutil.find_loader('timm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import GroupKFold
import timm

DATA_DIR = Path('/kaggle/input/csiro-biomass')
OUT_DIR = Path('/kaggle/working')
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('GPU is required for this experiment.')

SEED = 42
IMAGE_SIZE = int(os.getenv('IMAGE_SIZE', '384'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '16'))
EPOCHS = int(os.getenv('EPOCHS', '12'))
N_FOLDS = int(os.getenv('N_FOLDS', '5'))
BACKBONE = os.getenv('BACKBONE', 'convnextv2_tiny.fcmae_ft_in22k_in1k')
LR = float(os.getenv('LR', '2e-4'))
AUX_WEIGHT = float(os.getenv('AUX_WEIGHT', '0.25'))

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_everything(SEED)

TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
AUX_TARGETS = ['Pre_GSHH_NDVI', 'Height_Ave_cm']
WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5], dtype=np.float32)
print({'device': str(DEVICE), 'gpu': torch.cuda.get_device_name(0), 'backbone': BACKBONE})

In [ ]:
def load_train_wide():
    train = pd.read_csv(DATA_DIR / 'train.csv')
    meta_cols = [c for c in ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'] if c in train.columns]
    meta = train[['image_path'] + [c for c in meta_cols if c != 'image_path']].drop_duplicates('image_path')
    target = train.pivot_table(index='image_path', columns='target_name', values='target', aggfunc='first').reset_index()
    df = meta.merge(target, on='image_path', how='left')
    df['image_id'] = df['image_path'].astype(str)
    return df

train_df = load_train_wide()
print(train_df.shape)
display(train_df.head())
assert all(c in train_df.columns for c in TARGETS), TARGETS
assert all(c in train_df.columns for c in AUX_TARGETS), AUX_TARGETS

In [ ]:
class BiomassDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = DATA_DIR / str(row.image_path)
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.train and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        image = image.astype(np.float32) / 255.0
        image = (image - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        y = torch.tensor(row[TARGETS].values.astype(np.float32))
        aux = torch.tensor(row[AUX_TARGETS].values.astype(np.float32))
        return image, y, aux

class MultiTaskModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.encoder = timm.create_model(backbone, pretrained=True, num_classes=0, global_pool='avg')
        dim = self.encoder.num_features
        self.body = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.15), nn.Linear(dim, 512), nn.GELU(), nn.Dropout(0.1))
        self.biomass_head = nn.Linear(512, len(TARGETS))
        self.aux_head = nn.Linear(512, len(AUX_TARGETS))
    def forward(self, x):
        z = self.body(self.encoder(x))
        return self.biomass_head(z), self.aux_head(z)

def weighted_r2(y_true, y_pred):
    scores = []
    for i in range(len(TARGETS)):
        yt, yp = y_true[:, i], y_pred[:, i]
        den = np.sum((yt - yt.mean()) ** 2)
        scores.append(1.0 - np.sum((yt - yp) ** 2) / max(den, 1e-12))
    return float(np.sum(np.array(scores) * WEIGHTS)), dict(zip(TARGETS, scores))

def criterion(pred, target, aux_pred, aux_target):
    w = torch.tensor(WEIGHTS, device=target.device).view(1, -1)
    biomass = ((pred - target) ** 2 * w).mean()
    aux = nn.functional.smooth_l1_loss(aux_pred, aux_target)
    return biomass + AUX_WEIGHT * aux

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
oof = np.zeros((len(train_df), len(TARGETS)), dtype=np.float32)
logs = []

for fold, (trn_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['image_id'])):
    print(f'Fold {fold}')
    trn_ds = BiomassDataset(train_df.iloc[trn_idx], train=True)
    val_ds = BiomassDataset(train_df.iloc[val_idx], train=False)
    trn_dl = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)
    model = MultiTaskModel(BACKBONE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = GradScaler()
    best = -999
    best_pred = None
    for epoch in range(EPOCHS):
        model.train(); train_loss = 0.0
        for x, y, aux in trn_dl:
            x, y, aux = x.to(DEVICE), y.to(DEVICE), aux.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast():
                pred, aux_pred = model(x)
                loss = criterion(pred, y, aux_pred, aux)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            train_loss += loss.item() * len(x)
        sched.step()
        model.eval(); preds = []
        with torch.no_grad():
            for x, y, aux in val_dl:
                with autocast():
                    pred, _ = model(x.to(DEVICE))
                preds.append(pred.float().cpu().numpy())
        pred = np.concatenate(preds)
        score, per_target = weighted_r2(train_df.iloc[val_idx][TARGETS].values.astype(np.float32), pred)
        logs.append({'fold': fold, 'epoch': epoch, 'train_loss': train_loss / len(trn_ds), 'weighted_r2': score, **per_target})
        print(f'fold={fold} epoch={epoch} weighted_r2={score:.6f}')
        if score > best:
            best = score; best_pred = pred.copy()
            torch.save({'model': model.state_dict(), 'backbone': BACKBONE, 'image_size': IMAGE_SIZE, 'targets': TARGETS}, CKPT_DIR / f'fold{fold}.pt')
    oof[val_idx] = best_pred

oof_score, oof_detail = weighted_r2(train_df[TARGETS].values.astype(np.float32), oof)
print('OOF', oof_score, oof_detail)

In [ ]:
oof_df = train_df[['image_path']].copy()
for i, t in enumerate(TARGETS):
    oof_df[f'pred_{t}'] = oof[:, i]
    oof_df[f'true_{t}'] = train_df[t].values
oof_df.to_csv(OUT_DIR / 'oof_predictions.csv', index=False)
pd.DataFrame(logs).to_csv(OUT_DIR / 'training_log.csv', index=False)
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump({'cv_weighted_r2': oof_score, 'per_target_r2': oof_detail, 'config': {'backbone': BACKBONE, 'image_size': IMAGE_SIZE, 'epochs': EPOCHS, 'folds': N_FOLDS, 'aux_weight': AUX_WEIGHT}}, f, indent=2)
print('Saved OOF, logs, checkpoints, metrics.')

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = DATA_DIR / str(row.image_path)
        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        image = image.astype(np.float32) / 255.0
        image = (image - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
        return torch.from_numpy(image.transpose(2, 0, 1)).float(), row.image_path

test_long = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
test_images = test_long[['image_path']].drop_duplicates().reset_index(drop=True)
test_dl = DataLoader(TestDataset(test_images), batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)
test_pred = np.zeros((len(test_images), len(TARGETS)), dtype=np.float32)

for fold in range(N_FOLDS):
    ckpt = torch.load(CKPT_DIR / f'fold{fold}.pt', map_location='cpu')
    model = MultiTaskModel(ckpt['backbone']).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval(); fold_preds = []
    with torch.no_grad():
        for x, paths in test_dl:
            with autocast():
                p, _ = model(x.to(DEVICE))
            fold_preds.append(p.float().cpu().numpy())
    test_pred += np.concatenate(fold_preds) / N_FOLDS

pred_wide = test_images.copy()
for i, t in enumerate(TARGETS): pred_wide[t] = test_pred[:, i]
pred_long = pred_wide.melt(id_vars='image_path', value_vars=TARGETS, var_name='target_name', value_name='target')
sub = test_long[['sample_id', 'image_path', 'target_name']].merge(pred_long, on=['image_path', 'target_name'], how='left')[['sample_id', 'target']]
sub['target'] = sub['target'].clip(lower=0)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)
display(sub.head())